In [1]:
import subprocess
result = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
print(result.stdout)
for line in result.stdout.splitlines():
    if line.strip().endswith(" /"):
        use_pct = line.split()[4]
        print(f"Root disk usage: {use_pct}")
        if int(use_pct.rstrip('%')) >= 98:
            print("STILL FULL — clear space before continuing.")
        else:
            print("OK to proceed.")

Filesystem      Size  Used Avail Use% Mounted on
/dev/root       484G  452G   33G  94% /

Root disk usage: 94%
OK to proceed.


In [2]:
import sys, importlib
import numpy as np
from pathlib import Path

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "stage3_feature_extraction"))

import config3
importlib.reload(config3)
from stage3_utils import iter_epoch_files, ensure_all_output_dirs
ensure_all_output_dirs()

n_done = len(list((config3.STAGE3_OUT_ROOT / ".bootstrap_markers").glob("*.done")))
print(f"project_root = {project_root}")
print(f"Bootstrap markers so far: {n_done}")

project_root = /lp-dev/23BRS1328/erp_trial_min
Bootstrap markers so far: 6


In [3]:
bootstrap_path = project_root / "stage3_feature_extraction" / "bootstrap_grid.py"
text = bootstrap_path.read_text()

if "BATCH_SIZE = 50" in text:
    print("Already chunked — skipping.")
else:
    old_block = '''    if metric_fn is metric_mean_gfp:
        idx = rng.integers(0, n_trials, size=(n_iter, draw_size))
        sampled = epochs_data[idx]
        evoked = sampled.mean(axis=1)
        gfp = evoked.std(axis=1)
        values = gfp.mean(axis=1)
        return values.astype(np.float64)'''
    new_block = '''    if metric_fn is metric_mean_gfp:
        BATCH_SIZE = 50
        values = np.empty(n_iter, dtype=np.float64)
        for start in range(0, n_iter, BATCH_SIZE):
            end = min(start + BATCH_SIZE, n_iter)
            idx = rng.integers(0, n_trials, size=(end - start, draw_size))
            sampled = epochs_data[idx]
            evoked = sampled.mean(axis=1)
            gfp = evoked.std(axis=1)
            values[start:end] = gfp.mean(axis=1)
            del sampled, evoked, gfp
        return values.astype(np.float64)'''
    assert old_block in text, "Full-vectorized block not found — file may differ from what cell 89 wrote."
    bootstrap_path.write_text(text.replace(old_block, new_block))
    print("[Patched] chunked vectorization applied")

import bootstrap_grid
importlib.reload(bootstrap_grid)
print("Reloaded.")

Already chunked — skipping.
Reloaded.


In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/home/nvidia/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cuda
NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [2]:
import sys, importlib
import numpy as np
from pathlib import Path

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "stage3_feature_extraction"))

import config3
importlib.reload(config3)
from stage3_utils import iter_epoch_files, ensure_all_output_dirs
ensure_all_output_dirs()

n_done = len(list((config3.STAGE3_OUT_ROOT / ".bootstrap_markers").glob("*.done")))
print(f"Bootstrap markers so far: {n_done}")

Bootstrap markers so far: 6


In [3]:
bootstrap_path = project_root / "stage3_feature_extraction" / "bootstrap_grid.py"
text = bootstrap_path.read_text()

if "GPU_BATCH" in text:
    print("Already GPU-patched — skipping.")
else:
    new_func = '''def bootstrap_metric(
    epochs_data: np.ndarray,
    step,
    n_iter: int,
    rng: np.random.Generator,
    metric_fn: Callable[[np.ndarray], float] = METRIC_FN,
) -> np.ndarray:
    """
    GPU-accelerated (A100). Index draws stay on CPU via the same numpy
    Generator for identical sample selection vs the original loop-based
    version -- only the mean/std reduction runs on GPU.
    """
    n_trials = epochs_data.shape[0]
    draw_size = n_trials if step == "full" else int(step)

    if metric_fn is metric_mean_gfp:
        import torch
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        data_t = torch.as_tensor(epochs_data, dtype=torch.float32, device=dev)

        GPU_BATCH = 1000  # A100 has headroom; lower this if you hit OOM
        values = np.empty(n_iter, dtype=np.float64)
        for start in range(0, n_iter, GPU_BATCH):
            end = min(start + GPU_BATCH, n_iter)
            idx = rng.integers(0, n_trials, size=(end - start, draw_size))
            idx_t = torch.as_tensor(idx, dtype=torch.long, device=dev)

            sampled = data_t[idx_t]
            evoked = sampled.mean(dim=1)
            gfp = evoked.std(dim=1, unbiased=False)  # unbiased=False matches numpy's default ddof=0
            values[start:end] = gfp.mean(dim=1).cpu().numpy().astype(np.float64)
            del sampled, evoked, gfp, idx_t
        if dev.type == "cuda":
            torch.cuda.empty_cache()
        return values

    values = np.empty(n_iter, dtype=np.float64)
    for i in range(n_iter):
        idx = rng.integers(0, n_trials, size=draw_size)
        values[i] = metric_fn(epochs_data[idx])
    return values'''

    import re
    # Match either the original loop version or the cell-89 full-vectorized version
    pattern = re.compile(r'def bootstrap_metric\(.*?\n    return values', re.DOTALL)
    assert pattern.search(text), "Could not locate bootstrap_metric — check bootstrap_grid.py manually."
    text = pattern.sub(new_func, text, count=1)
    bootstrap_path.write_text(text)
    print("[Patched] GPU-accelerated bootstrap_metric written to disk")

import bootstrap_grid
importlib.reload(bootstrap_grid)
print("Reloaded.")

[Patched] GPU-accelerated bootstrap_metric written to disk
Reloaded.


In [4]:
import logging, csv, time
from collections import defaultdict

logging.getLogger("phase32_bootstrap").setLevel(logging.CRITICAL)
MARKER_DIR = config3.STAGE3_OUT_ROOT / ".bootstrap_markers"
MARKER_DIR.mkdir(parents=True, exist_ok=True)

refs_by_paradigm = defaultdict(list)
for ref in iter_epoch_files():
    refs_by_paradigm[f"{ref.dataset}_{ref.task}"].append(ref)

paradigms_remaining = [p for p in refs_by_paradigm if not (MARKER_DIR / f"{p}.done").exists()]
total_files_remaining = sum(len(refs_by_paradigm[p]) for p in paradigms_remaining)
print(f"Paradigms remaining: {len(paradigms_remaining)} | Files remaining: {total_files_remaining}")

rng = np.random.default_rng(config3.RANDOM_SEED)
files_done_total = 0
t_start = time.time()

for p_idx, paradigm in enumerate(paradigms_remaining):
    refs = refs_by_paradigm[paradigm]
    marker = MARKER_DIR / f"{paradigm}.done"
    out_csv = config3.BOOTSTRAP_DIR / f"{paradigm}_bootstrap.csv"
    if out_csv.exists():
        out_csv.unlink()

    t0 = time.time()
    n_done = 0
    for ref in refs:
        try:
            rows = bootstrap_grid.process_file(ref, rng)
        except Exception as exc:
            print(f"  [ERROR] {ref.stem}: {exc}")
            continue
        if not rows:
            continue
        need_header = (not out_csv.exists()) or out_csv.stat().st_size == 0
        with out_csv.open("a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=bootstrap_grid.FIELDNAMES)
            if need_header:
                writer.writeheader()
            for row in rows:
                writer.writerow(row)
        n_done += 1

    files_done_total += n_done
    marker.touch()
    elapsed_this = time.time() - t0
    rate = files_done_total / (time.time() - t_start)  # files/sec, cumulative
    eta_hours = (total_files_remaining - files_done_total) / rate / 3600 if rate > 0 else float("nan")
    print(f"[{p_idx+1}/{len(paradigms_remaining)}] {paradigm}: {n_done}/{len(refs)} files, "
          f"{elapsed_this:.1f}s ({elapsed_this/max(n_done,1):.2f}s/file) | ETA remaining: {eta_hours:.1f}h")

print("\nBootstrap regeneration complete.")

Paradigms remaining: 18 | Files remaining: 1352


/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:128: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = epochs.get_data(picks=picks)  # (n_trials, n_channels, n_times)
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:145: RuntimeWarning: Mean of empty slice
  "metric_mean": float(np.nanmean(values)),
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:128: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is e

[1/18] ERP_CORE_P3: 109/109 files, 259.1s (2.38s/file) | ETA remaining: 0.8h


/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:128: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = epochs.get_data(picks=picks)  # (n_trials, n_channels, n_times)
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:145: RuntimeWarning: Mean of empty slice
  "metric_mean": float(np.nanmean(values)),
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


[2/18] DS006018_auditoryoddball: 340/340 files, 406.9s (1.20s/file) | ETA remaining: 0.4h
[3/18] DS006018_flanker: 70/70 files, 132.5s (1.89s/file) | ETA remaining: 0.4h
[4/18] DS006018_visualoddball: 95/95 files, 133.0s (1.40s/file) | ETA remaining: 0.3h
[5/18] DS006018_visualsearch: 69/69 files, 125.7s (1.82s/file) | ETA remaining: 0.3h


...

/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:128: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = epochs.get_data(picks=picks)  # (n_trials, n_channels, n_times)
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:145: RuntimeWarning: Mean of empty slice
  "metric_mean": float(np.nanmean(values)),
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


[16/18] DS002680_gonogo_7: 57/57 files, 136.9s (2.40s/file) | ETA remaining: 0.1h


/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:128: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = epochs.get_data(picks=picks)  # (n_trials, n_channels, n_times)
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:145: RuntimeWarning: Mean of empty slice
  "metric_mean": float(np.nanmean(values)),
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


[17/18] DS002680_gonogo_8: 51/51 files, 127.4s (2.50s/file) | ETA remaining: 0.0h
[18/18] DS002680_gonogo_9: 51/51 files, 127.0s (2.49s/file) | ETA remaining: 0.0h

Bootstrap regeneration complete.


/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:128: RuntimeWarning: epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.
  data = epochs.get_data(picks=picks)  # (n_trials, n_channels, n_times)
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:145: RuntimeWarning: Mean of empty slice
  "metric_mean": float(np.nanmean(values)),
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [5]:
# Verify all 24 paradigms are truly marked done
MARKER_DIR = config3.STAGE3_OUT_ROOT / ".bootstrap_markers"
done_markers = sorted(p.stem for p in MARKER_DIR.glob("*.done"))
print(f"Markers found: {len(done_markers)} / 24")
for m in done_markers:
    print(" ", m)

missing = [p for p in refs_by_paradigm if not (MARKER_DIR / f"{p}.done").exists()]
print(f"\nStill missing: {missing}")

Markers found: 24 / 24
  DS002680_gonogo_1
  DS002680_gonogo_10
  DS002680_gonogo_11
  DS002680_gonogo_12
  DS002680_gonogo_13
  DS002680_gonogo_2
  DS002680_gonogo_3
  DS002680_gonogo_4
  DS002680_gonogo_5
  DS002680_gonogo_6
  DS002680_gonogo_7
  DS002680_gonogo_8
  DS002680_gonogo_9
  DS006018_auditoryoddball
  DS006018_flanker
  DS006018_visualoddball
  DS006018_visualsearch
  ERP_CORE_ERN
  ERP_CORE_LRP
  ERP_CORE_MMN
  ERP_CORE_N170
  ERP_CORE_N2pc
  ERP_CORE_N400
  ERP_CORE_P3

Still missing: []


In [6]:
import pandas as pd

nan_report = []
for csv_path in sorted(config3.BOOTSTRAP_DIR.glob("*_bootstrap.csv")):
    df = pd.read_csv(csv_path)
    if "metric_mean" in df.columns:
        bad = df["metric_mean"].isna().sum()
        if bad:
            nan_report.append((csv_path.name, bad, len(df)))

if nan_report:
    print("Paradigms with NaN rows:")
    for name, n_bad, n_total in nan_report:
        print(f"  {name}: {n_bad}/{n_total} rows NaN")
else:
    print("No NaN rows found.")

Paradigms with NaN rows:
  DS002680_gonogo_10_bootstrap.csv: 1/531 rows NaN
  DS002680_gonogo_11_bootstrap.csv: 1/501 rows NaN
  DS002680_gonogo_12_bootstrap.csv: 1/451 rows NaN
  DS002680_gonogo_13_bootstrap.csv: 1/501 rows NaN
  DS002680_gonogo_1_bootstrap.csv: 1/471 rows NaN
  DS002680_gonogo_2_bootstrap.csv: 1/491 rows NaN
  DS002680_gonogo_3_bootstrap.csv: 1/511 rows NaN
  DS002680_gonogo_4_bootstrap.csv: 1/501 rows NaN
  DS002680_gonogo_5_bootstrap.csv: 1/511 rows NaN
  DS002680_gonogo_6_bootstrap.csv: 1/541 rows NaN
  DS002680_gonogo_7_bootstrap.csv: 1/561 rows NaN
  DS002680_gonogo_8_bootstrap.csv: 1/501 rows NaN
  DS002680_gonogo_9_bootstrap.csv: 1/501 rows NaN
  DS006018_auditoryoddball_bootstrap.csv: 1/2441 rows NaN
  ERP_CORE_LRP_bootstrap.csv: 2/1062 rows NaN
  ERP_CORE_P3_bootstrap.csv: 2/1442 rows NaN


In [7]:
import subprocess

# What run_rf.py currently expects/does
print("=== run_rf.py ===")
print((project_root / "run_rf.py").read_text()[:2000])

print("\n=== feature_matrices/ layout ===")
fm_dir = project_root / "stage3_results" / "feature_matrices"
for p in sorted(fm_dir.rglob("*"))[:30]:
    print(" ", p.relative_to(fm_dir))

print("\n=== does stage4_classical_ml/ exist? ===")
print((project_root / "stage4_classical_ml").exists())

=== run_rf.py ===
"""
Random Forest Baseline
Runs LOSO on every trial reduction step.
"""

from pathlib import Path

import numpy as np
import pandas as pd

from stage4_classical_ml.dataset import DatasetLoader
from stage4_classical_ml.loso_cv import LOSOCV
from stage4_classical_ml.rf_model import RFModel


DATASET = "ERP_CORE__P3"

OUTPUT_DIR = Path("results_stage4/random_forest")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


loader = DatasetLoader()

steps = loader.available_steps(DATASET)

summary = []

print("=" * 80)
print(DATASET)
print("=" * 80)

for step in steps:

    print(f"\nTrial Step : {step}")

    dataset = loader.load(DATASET, step=step)

    loso = LOSOCV()

    rf = RFModel()

    fold_acc = []
    fold_f1 = []

    for subject, train_idx, test_idx in loso.split(dataset.metadata):

        X_train = dataset.X[train_idx]
        X_test = dataset.X[test_idx]

        y_train = dataset.y[train_idx]
        y_test = dataset.y[test_idx]

        y_pred = rf.train_predic

In [8]:
print("=== Does feature_matrices/ exist at all? ===")
fm_dir = project_root / "stage3_results" / "feature_matrices"
print(fm_dir, "exists:", fm_dir.exists())

print("\n=== stage3_results/ top-level contents ===")
sr_dir = project_root / "stage3_results"
for p in sorted(sr_dir.iterdir()):
    print(" ", p.name, "(dir)" if p.is_dir() else "(file)")

print("\n=== Recursive search for anything matrix/matrices-like ===")
for p in sr_dir.rglob("*"):
    if "matrix" in p.name.lower() or "matrices" in p.name.lower():
        print(" ", p.relative_to(sr_dir))

print("\n=== What's actually in stage3_results/ml/ (mentioned as having prior ML outputs) ===")
ml_dir = sr_dir / "ml"
if ml_dir.exists():
    for p in sorted(ml_dir.rglob("*"))[:40]:
        print(" ", p.relative_to(ml_dir))
else:
    print("  does not exist")

print("\n=== stage3_feature_extraction/ contents (for assemble_features_binary.py output path) ===")
sfe_dir = project_root / "stage3_feature_extraction"
for p in sorted(sfe_dir.iterdir()):
    print(" ", p.name)

=== Does feature_matrices/ exist at all? ===
/lp-dev/23BRS1328/erp_trial_min/stage3_results/feature_matrices exists: True

=== stage3_results/ top-level contents ===
  .bootstrap_markers (dir)
  audit (dir)
  bootstrap (dir)
  feature_matrices (dir)
  features (dir)
  logs (dir)
  ml (dir)
  subsample_indices (dir)

=== Recursive search for anything matrix/matrices-like ===
  feature_matrices

=== What's actually in stage3_results/ml/ (mentioned as having prior ML outputs) ===
  DS002680__gonogo_1
  DS002680__gonogo_1/X.npy
  DS002680__gonogo_1/feature_block_boundaries.json
  DS002680__gonogo_1/metadata.csv
  DS002680__gonogo_1/y.npy
  DS002680__gonogo_10
  DS002680__gonogo_10/X.npy
  DS002680__gonogo_10/feature_block_boundaries.json
  DS002680__gonogo_10/metadata.csv
  DS002680__gonogo_10/y.npy
  DS002680__gonogo_11
  DS002680__gonogo_11/X.npy
  DS002680__gonogo_11/feature_block_boundaries.json
  DS002680__gonogo_11/metadata.csv
  DS002680__gonogo_11/y.npy
  DS002680__gonogo_12
  DS00

In [9]:
# ============================================================
# DIAGNOSTIC CELL: Actual Phase 3.2 bootstrap completion status
# ============================================================
from pathlib import Path

bootstrap_dir = Path("/lp-dev/23BRS1328/erp_trial_min/stage3_results/bootstrap")
markers_dir = Path("/lp-dev/23BRS1328/erp_trial_min/stage3_results/.bootstrap_markers")

bootstrap_csvs = sorted(bootstrap_dir.glob("*.csv"))
markers = sorted(markers_dir.glob("*.done"))

print(f"Bootstrap CSV files: {len(bootstrap_csvs)}")
for f in bootstrap_csvs:
    print(f"  {f.name}")

print(f"\nCompletion markers: {len(markers)}")
for f in markers:
    print(f"  {f.name}")

print(f"\nExpected total paradigms: 24")
print(f"Bootstrap complete: {len(markers) == 24}")

Bootstrap CSV files: 24
  DS002680_gonogo_10_bootstrap.csv
  DS002680_gonogo_11_bootstrap.csv
  DS002680_gonogo_12_bootstrap.csv
  DS002680_gonogo_13_bootstrap.csv
  DS002680_gonogo_1_bootstrap.csv
  DS002680_gonogo_2_bootstrap.csv
  DS002680_gonogo_3_bootstrap.csv
  DS002680_gonogo_4_bootstrap.csv
  DS002680_gonogo_5_bootstrap.csv
  DS002680_gonogo_6_bootstrap.csv
  DS002680_gonogo_7_bootstrap.csv
  DS002680_gonogo_8_bootstrap.csv
  DS002680_gonogo_9_bootstrap.csv
  DS006018_auditoryoddball_bootstrap.csv
  DS006018_flanker_bootstrap.csv
  DS006018_visualoddball_bootstrap.csv
  DS006018_visualsearch_bootstrap.csv
  ERP_CORE_ERN_bootstrap.csv
  ERP_CORE_LRP_bootstrap.csv
  ERP_CORE_MMN_bootstrap.csv
  ERP_CORE_N170_bootstrap.csv
  ERP_CORE_N2pc_bootstrap.csv
  ERP_CORE_N400_bootstrap.csv
  ERP_CORE_P3_bootstrap.csv

Completion markers: 24
  DS002680_gonogo_1.done
  DS002680_gonogo_10.done
  DS002680_gonogo_11.done
  DS002680_gonogo_12.done
  DS002680_gonogo_13.done
  DS002680_gonogo_2.d

In [10]:
# ============================================================
# DIAGNOSTIC CELL: Sanity check bootstrap CSV contents
# ============================================================
import pandas as pd

bootstrap_dir = Path("/lp-dev/23BRS1328/erp_trial_min/stage3_results/bootstrap")

for name in ["ERP_CORE_ERN_bootstrap.csv", "ERP_CORE_MMN_bootstrap.csv", "DS006018_flanker_bootstrap.csv"]:
    df = pd.read_csv(bootstrap_dir / name)
    print(f"\n{name}")
    print(f"  shape: {df.shape}")
    print(f"  columns: {list(df.columns)}")
    print(f"  unique steps: {sorted(df['step'].unique(), key=str) if 'step' in df.columns else 'N/A'}")
    print(f"  n_iterations values: {df['n_iterations'].unique() if 'n_iterations' in df.columns else 'N/A'}")
    print(df.head(3).to_string())


ERP_CORE_ERN_bootstrap.csv
  shape: (740, 10)
  columns: ['dataset', 'subject', 'task', 'step', 'n_total_epochs', 'n_iterations', 'metric_mean', 'metric_std', 'ci_low', 'ci_high']
  unique steps: ['10', '100', '150', '20', '200', '30', '5', '50', '75', 'full']
  n_iterations values: [1000]
    dataset  subject task step  n_total_epochs  n_iterations  metric_mean    metric_std    ci_low   ci_high
0  ERP_CORE  sub-001  ERN    5              88          1000     0.000005  4.673679e-07  0.000004  0.000006
1  ERP_CORE  sub-001  ERN   10              88          1000     0.000004  3.582158e-07  0.000003  0.000004
2  ERP_CORE  sub-001  ERN   20              88          1000     0.000003  2.694433e-07  0.000003  0.000004

ERP_CORE_MMN_bootstrap.csv
  shape: (1480, 10)
  columns: ['dataset', 'subject', 'task', 'step', 'n_total_epochs', 'n_iterations', 'metric_mean', 'metric_std', 'ci_low', 'ci_high']
  unique steps: ['10', '100', '150', '20', '200', '30', '5', '50', '75', 'full']
  n_iteration

In [11]:
# ============================================================
# DIAGNOSTIC CELL: Check how widespread the near-zero-trial pattern is in MMN
# ============================================================
df_mmn = pd.read_csv(bootstrap_dir / "ERP_CORE_MMN_bootstrap.csv")

# Get n_total_epochs per subject (should be same across all steps for a subject)
per_subject = df_mmn.groupby("subject")["n_total_epochs"].first().sort_values()
print("Trial counts per subject (lowest first):")
print(per_subject.to_string())

print(f"\nSubjects with fewer than 30 total epochs: {(per_subject < 30).sum()} / {len(per_subject)}")
print(f"Median trial count: {per_subject.median()}")

Trial counts per subject (lowest first):
subject
sub-007      1
sub-012     10
sub-039     38
sub-030     55
sub-032     61
sub-008     74
sub-017     82
sub-001     96
sub-040    102
sub-010    111
sub-035    111
sub-018    112
sub-009    117
sub-014    120
sub-037    126
sub-021    127
sub-029    128
sub-036    134
sub-006    142
sub-031    142
sub-016    147
sub-028    149
sub-003    154
sub-022    155
sub-025    159
sub-015    163
sub-002    164
sub-024    164
sub-027    167
sub-011    167
sub-038    167
sub-013    168
sub-026    171
sub-020    177
sub-005    181
sub-034    186
sub-023    192
sub-004    197
sub-033    199
sub-019    200

Subjects with fewer than 30 total epochs: 2 / 40
Median trial count: 144.5


In [12]:
# ============================================================
# STARTER DIAGNOSTIC: run before starting the new chat.
# Paste this output along with the handoff prompt above.
# ============================================================
from pathlib import Path

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")

print("=" * 60)
print("run_rf.py -- full contents (the script referencing the")
print("missing stage4_classical_ml module)")
print("=" * 60)
rf_path = project_root / "run_rf.py"
if rf_path.exists():
    print(rf_path.read_text())
else:
    print("NOT FOUND")

print("\n" + "=" * 60)
print("stage4_classical_ml/ -- does it exist at all?")
print("=" * 60)
s4_path = project_root / "stage4_classical_ml"
print(f"Exists: {s4_path.exists()}")
if s4_path.exists():
    for f in sorted(s4_path.rglob("*.py")):
        print(f"  {f.relative_to(project_root)}")

print("\n" + "=" * 60)
print("stage3_results/ml/ -- confirm all 20 in-scope paradigms present")
print("=" * 60)
ml_dir = project_root / "stage3_results" / "ml"
folders = sorted([d.name for d in ml_dir.iterdir() if d.is_dir()])
print(f"Total folders: {len(folders)}")
for f in folders:
    print(f"  {f}")

print("\n" + "=" * 60)
print("Sample: shapes and label check for one ERP_CORE and one DS002680 paradigm")
print("=" * 60)
import numpy as np
for sample in ["ERP_CORE__P3", "DS002680__gonogo_1"]:
    d = ml_dir / sample
    if d.exists():
        X = np.load(d / "X.npy")
        y = np.load(d / "y.npy")
        print(f"{sample}: X={X.shape}, y={y.shape}, classes={np.unique(y)}, "
              f"class_balance={np.bincount(y).tolist()}")
    else:
        print(f"{sample}: NOT FOUND")

print("\n" + "=" * 60)
print("subsample_indices/ -- confirm 10-step grid present, sample count")
print("=" * 60)
idx_dir = project_root / "stage3_results" / "subsample_indices"
all_json = list(idx_dir.rglob("*.json"))
print(f"Total index files: {len(all_json)}")
if all_json:
    import json
    with open(all_json[0]) as f:
        sample_record = json.load(f)
    print(f"Sample file steps: {list(sample_record['steps'].keys())}")
    print(f"Sample record keys: {list(sample_record.keys())}")

print("\n" + "=" * 60)
print("NFS_benchmarks.csv -- confirm structure")
print("=" * 60)
import pandas as pd
nfs_path = project_root / "data" / "benchmarks" / "NFS_benchmarks.csv"
if nfs_path.exists():
    df_nfs = pd.read_csv(nfs_path)
    print(f"Shape: {df_nfs.shape}")
    print(f"Columns (first 10): {list(df_nfs.columns[:10])}")
    print(f"Paradigms: {df_nfs.iloc[:,0].tolist() if df_nfs.shape[1] > 0 else 'N/A'}")
else:
    print("NOT FOUND")

run_rf.py -- full contents (the script referencing the
missing stage4_classical_ml module)
"""
Random Forest Baseline
Runs LOSO on every trial reduction step.
"""

from pathlib import Path

import numpy as np
import pandas as pd

from stage4_classical_ml.dataset import DatasetLoader
from stage4_classical_ml.loso_cv import LOSOCV
from stage4_classical_ml.rf_model import RFModel


DATASET = "ERP_CORE__P3"

OUTPUT_DIR = Path("results_stage4/random_forest")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


loader = DatasetLoader()

steps = loader.available_steps(DATASET)

summary = []

print("=" * 80)
print(DATASET)
print("=" * 80)

for step in steps:

    print(f"\nTrial Step : {step}")

    dataset = loader.load(DATASET, step=step)

    loso = LOSOCV()

    rf = RFModel()

    fold_acc = []
    fold_f1 = []

    for subject, train_idx, test_idx in loso.split(dataset.metadata):

        X_train = dataset.X[train_idx]
        X_test = dataset.X[test_idx]

        y_train = dataset.y[train_id

In [13]:
from pathlib import Path
import json, pandas as pd

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")

print("=== metadata.csv columns/head (ERP_CORE__P3) ===")
md = pd.read_csv(project_root / "stage3_results/ml/ERP_CORE__P3/metadata.csv")
print(md.columns.tolist())
print(md.head())
print(f"unique subjects: {md['subject'].nunique()}")

print("\n=== feature_block_boundaries.json (ERP_CORE__P3) ===")
print((project_root / "stage3_results/ml/ERP_CORE__P3/feature_block_boundaries.json").read_text())

print("\n=== one full subsample_indices record ===")
idx_files = list((project_root / "stage3_results/subsample_indices").rglob("*.json"))
print(json.dumps(json.loads(idx_files[0].read_text()), indent=2)[:2000])

print("\n=== bootstrap_grid.py location + signature-relevant lines ===")
bg = list(project_root.rglob("bootstrap_grid.py"))
print(bg)
if bg:
    print(bg[0].read_text()[:3000])

print("\n=== env check ===")
import importlib
for pkg in ["sklearn", "xgboost", "scipy"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg}: {getattr(m, '__version__', 'unknown')}")
    except ImportError as e:
        print(f"{pkg}: MISSING ({e})")

=== metadata.csv columns/head (ERP_CORE__P3) ===
['row_index', 'subject', 'file_id', 'epoch_index_in_subject', 'label']
   row_index  subject       file_id  epoch_index_in_subject  label
0          0  sub-001  standard-epo                       0      1
1          1  sub-001  standard-epo                       1      1
2          2  sub-001  standard-epo                       2      1
3          3  sub-001  standard-epo                       3      1
4          4  sub-001  standard-epo                       4      1
unique subjects: 36

=== feature_block_boundaries.json (ERP_CORE__P3) ===
{
  "band_power": [
    0,
    150
  ],
  "plv_alpha": [
    150,
    585
  ],
  "plv_beta": [
    585,
    1020
  ],
  "wpli_alpha": [
    1020,
    1455
  ],
  "wpli_beta": [
    1455,
    1890
  ],
  "entropy": [
    1890,
    1920
  ],
  "_total_features": 1920
}

=== one full subsample_indices record ===
{
  "dataset": "ERP_CORE",
  "subject": "sub-001",
  "task": "ERN",
  "file_id": "correct-epo